# Configuração do Servidor Ollama e Modelos Locais

Esta seção inicializa o servidor local Ollama e realiza o download dos modelos necessários para embeddings (`nomic-embed-text`) e geração de respostas (`qwen2.5:3b`).

In [19]:
# Installing ollama
!apt-get install -y -qq zstd > /dev/null 2>&1 || (apt-get update -qq > /dev/null 2>&1 && apt-get install -y -qq zstd > /dev/null 2>&1)
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -n 2

>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [20]:
# Running ollama server
import subprocess
import time
import requests

OLLAMA_URL = "http://localhost:11434"

def server_is_up():
    try:
        return requests.get(f"{OLLAMA_URL}/api/version", timeout=2).ok
    except requests.RequestException:
        return False

if not server_is_up():
    subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(30):
        if server_is_up():
            break
        time.sleep(1)

print("Server running:", server_is_up())

Server running: True


In [21]:
# Pull models
!ollama pull qwen2.5:3b
!ollama pull nomic-embed-text
!ollama list



NAME                       ID              SIZE      MODIFIED               
nomic-embed-text:latest    0a109f422b47    274 MB    Less than a second ago    
qwen2.5:3b                 357c53fb659c    1.9 GB    1 second ago              


# Dependências, Configuração de Caminhos e Provisionamento do Corpus

Instalação dos pacotes necessários (`chromadb`, `openai`, `tiktoken`, `langchain-text-splitters`), configuração de caminhos persistentes (Google Drive ou local) e provisionamento resiliente dos 10 manuais técnicos da Huawei Ascend.

In [22]:
# Installing dependencies
!pip -q install chromadb openai tiktoken pandas pyarrow tqdm requests langchain-text-splitters

In [23]:
import os, re, glob, json, math, time, shutil, urllib.request
from pathlib import Path
from collections import defaultdict, Counter
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from openai import OpenAI
import chromadb

# ============================================================================
# Google Drive & Local Directory Configuration
# ============================================================================
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE_DIR = Path("/content/drive/MyDrive/RAG_Atlas200DK")
    except Exception:
        BASE_DIR = Path(".")
else:
    BASE_DIR = Path(".")

PROCESSED_MD_DIR = BASE_DIR / "data" / "processed" / "markdown"
OUTPUT_DIR       = BASE_DIR / "data" / "output"
CHROMA_DIR       = str(BASE_DIR / "chroma_db")

for d in [PROCESSED_MD_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

OLLAMA_URL = "http://localhost:11434"
GEN_MODEL  = "qwen2.5:3b"
EMB_MODEL  = "nomic-embed-text"
COLLECTION_NAME = "huawei_ascend_atomic_col"

client = OpenAI(base_url=f"{OLLAMA_URL}/v1", api_key="ollama")

# ============================================================================
# Resilient Corpus Asset Provisioning (10 Markdown Manuals + Chunk Dataset)
# ============================================================================
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/ramonvfs/RAG_Atlas200DK/master/"
CHUNK_DATASET_STEM = "huawei_ascend_rag_chunks_atomic"

EXPECTED_DOCS = [
    "ApplicationDevelopmentGuide_MindStudio_.md",
    "ATCToolInstructions.md",
    "Atlas200AIAcceleratorModuleApplicationSoftwareDevelopmentGuide_Models3000_.md",
    "EnvironmentDeploymentGuide.md",
    "Huawei_Atlas_200_DK_developer_kit_Technical_White_Paper__Model_3000__08.md",
    "MindStudioInstallationGuideUbuntuX86.md",
    "TBECustomOperatorDevelopmentGuide_CLI_.md",
    "TensorFlowNetworkModelPortingandTrainingGuide.md",
    "UserGuide2020.md",
    "UserGuide2021.md",
]

ALT_ROOTS = [
    Path("."),
    Path("/content/RAG_Atlas200DK"),
    Path("/content"),
    Path("/content/drive/MyDrive/RAG_Atlas200DK"),
]

def _download_file(rel_path: str, dest: Path) -> bool:
    url = GITHUB_RAW_BASE + urllib.request.quote(rel_path)
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=60) as resp, open(dest, "wb") as out_f:
            out_f.write(resp.read())
        print(f"  [DOWNLOADED] {rel_path}")
        return True
    except Exception as e:
        print(f"  [WARNING] Could not download {rel_path}: {e}")
        return False

def ensure_corpus_assets(md_dir: Path, out_dir: Path) -> tuple:
    # 1. Ensure Markdown manuals exist
    for name in EXPECTED_DOCS:
        target = md_dir / name
        if target.exists():
            continue
        for root in ALT_ROOTS:
            cand = root / "data" / "processed" / "markdown" / name
            if cand.exists() and cand.resolve() != target.resolve():
                shutil.copy2(cand, target)
                print(f"  [SYNCED] {name} from {cand.parent}")
                break
        if not target.exists():
            _download_file(f"data/processed/markdown/{name}", target)

    md_paths = sorted(p for p in md_dir.glob("*.md") if p.is_file())

    # 2. Check for pre-computed chunk dataset (parquet preferred, then json)
    dataset_path = None
    for ext in [".parquet", ".json"]:
        target = out_dir / f"{CHUNK_DATASET_STEM}{ext}"
        if target.exists():
            dataset_path = target
            break
        for root in ALT_ROOTS:
            cand = root / "data" / "output" / f"{CHUNK_DATASET_STEM}{ext}"
            if cand.exists() and cand.resolve() != target.resolve():
                shutil.copy2(cand, target)
                print(f"  [SYNCED] {target.name} from {cand.parent}")
                dataset_path = target
                break
        if dataset_path:
            break

    if dataset_path is None:
        for ext in [".parquet", ".json"]:
            target = out_dir / f"{CHUNK_DATASET_STEM}{ext}"
            if _download_file(f"data/output/{CHUNK_DATASET_STEM}{ext}", target):
                dataset_path = target
                break

    return md_paths, dataset_path

md_paths, chunk_dataset_path = ensure_corpus_assets(PROCESSED_MD_DIR, OUTPUT_DIR)
print(f"[OK] Located {len(md_paths)} Markdown manuals in {PROCESSED_MD_DIR}")
if chunk_dataset_path:
    print(f"[OK] Found pre-computed chunk dataset: {chunk_dataset_path.name}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[OK] Located 10 Markdown manuals in /content/drive/MyDrive/RAG_Atlas200DK/data/processed/markdown
[OK] Found pre-computed chunk dataset: huawei_ascend_rag_chunks_atomic.parquet


# Estratégias de Chunking: Atômico Preservando Tabelas (Ativo) vs Recursivo (Linha de Base)

Aqui definimos a **Estratégia 3 (Atomic Table-Aware)** como particionador ativo do corpus:
- Detecta hierarquias de seções (`#`, `##`, `###`, `####`) e injeta breadcrumbs hierárquicos `[Doc: <doc> > Section: <caminho>]`.
- Trata tabelas de especificação (`| ... |`) e blocos de código (```` ``` ````) como blocos atomicos indivisíveis.
- Sub-divide apenas prosa extensa excedendo `max_chunk_tokens` usando `RecursiveCharacterTextSplitter`.

A **Estratégia 1 (Recursive Window)** e mantida implementada, mas comentada para benchmarking futuro.

In [24]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Tokenizer wrapper using OpenAI cl100k_base
_ENC = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(_ENC.encode(text))

# ============================================================================
# Strategy 3: Atomic Table-Aware & Breadcrumb Hybrid Chunking (ACTIVE)
# ============================================================================
def strategy_atomic_table_aware(
    text: str,
    doc_name: str = "unknown",
    max_chunk_tokens: int = 1024,
    chunk_overlap: int = 64,
) -> list[dict]:
    """
    Strategy 3: Atomic table-aware chunking with breadcrumb context.

    Pipeline:
    1. Parse the heading hierarchy (#, ##, ###, ####) to build section paths.
    2. Within each section, identify atomic blocks (tables, code fences).
    3. Keep atomic blocks intact; sub-split prose exceeding max_chunk_tokens.
    4. Prepend breadcrumb path to every chunk for standalone readability.
    """
    lines = text.split("\n")
    sections = []
    current_headers = {}
    current_lines = []
    current_has_table = False

    def _flush_section():
        nonlocal current_lines, current_has_table
        if current_lines:
            path_parts = [
                current_headers[k]
                for k in sorted(current_headers.keys())
            ]
            sections.append((
                " > ".join(path_parts) if path_parts else "Root",
                "\n".join(current_lines),
                current_has_table,
            ))
        current_lines = []
        current_has_table = False

    for line in lines:
        header_match = re.match(r"^(#{1,4})\s+(.+)$", line)
        if header_match:
            _flush_section()
            level = len(header_match.group(1))
            title = header_match.group(2).strip()
            current_headers[level] = title
            for k in list(current_headers.keys()):
                if k > level:
                    del current_headers[k]
            current_lines.append(line)
        else:
            current_lines.append(line)
            if line.strip().startswith("|"):
                current_has_table = True
    _flush_section()

    results = []
    for header_path, section_text, _ in sections:
        breadcrumb = f"[Doc: {doc_name} > {header_path}]"

        parts = re.split(
            r"(```[\s\S]*?```|(?:^\|.+\|$\n?){2,})",
            section_text,
            flags=re.MULTILINE,
        )

        for part in parts:
            part_stripped = part.strip()
            if not part_stripped:
                continue

            is_table = part_stripped.startswith("|")
            is_code = part_stripped.startswith("```")
            is_atomic = is_table or is_code

            chunk_text = f"{breadcrumb}\n\n{part_stripped}"
            tok_count = count_tokens(chunk_text)

            if is_atomic or tok_count <= max_chunk_tokens:
                results.append({
                    "text": chunk_text,
                    "strategy": "atomic_table_aware",
                    "chunk_index": len(results),
                    "approx_tokens": tok_count,
                    "metadata": {
                        "doc_name": doc_name,
                        "header_path": header_path,
                        "is_table": is_table,
                        "is_code_block": is_code,
                    }
                })
            else:
                sub_splitter = RecursiveCharacterTextSplitter(
                    chunk_size=max_chunk_tokens,
                    chunk_overlap=chunk_overlap,
                    length_function=count_tokens,
                )
                for sc in sub_splitter.split_text(part_stripped):
                    sc_with_breadcrumb = f"{breadcrumb}\n\n{sc}"
                    results.append({
                        "text": sc_with_breadcrumb,
                        "strategy": "atomic_table_aware",
                        "chunk_index": len(results),
                        "approx_tokens": count_tokens(sc_with_breadcrumb),
                        "metadata": {
                            "doc_name": doc_name,
                            "header_path": header_path,
                            "is_table": False,
                            "is_code_block": False,
                        }
                    })

    return results

print("[OK] Strategy 3 (Atomic Table-Aware) defined")

# ============================================================================
# Strategy 1: Fixed/Recursive Window Splitting (COMMENTED OUT FOR COMPARISON)
# ============================================================================
# def strategy_recursive_split(
#     text: str,
#     doc_name: str = "unknown",
#     chunk_size: int = 1024,
#     chunk_overlap: int = 128,
# ) -> list[dict]:
#     """
#     Strategy 1: Fixed/recursive window splitting (baseline).
#     Splits text based on character separators, ignoring table boundaries.
#     """
#     splitter = RecursiveCharacterTextSplitter(
#         chunk_size=chunk_size,
#         chunk_overlap=chunk_overlap,
#         length_function=count_tokens,
#         separators=["\n\n", "\n", ". ", " ", ""],
#     )
#     chunks = splitter.split_text(text)
#     return [
#         {
#             "text": chunk,
#             "strategy": "recursive",
#             "chunk_index": i,
#             "approx_tokens": count_tokens(chunk),
#             "metadata": {
#                 "doc_name": doc_name,
#                 "header_path": "Root",
#                 "is_table": False,
#                 "is_code_block": False,
#             }
#         }
#         for i, chunk in enumerate(chunks)
#     ]

[OK] Strategy 3 (Atomic Table-Aware) defined


# Carregamento dos Manuais e Padronização dos Chunks

Esta célula carrega os chunks atômicos gerados a partir dos 10 manuais. Se o arquivo serializado `huawei_ascend_rag_chunks_atomic.parquet` (ou `.json`) estiver disponível em `data/output/`, ele e carregado diretamente (fast-path em < 2 segundos). Caso contrário, o notebook processa todos os manuais em lote usando `strategy_atomic_table_aware`.

In [25]:
LOAD_PRECOMPUTED_CHUNKS = True
_HTML_TAG_RE = re.compile(r"</?(?:span|sup|sub|br|b|i|u|p|div)\b[^>]*>", re.I)
_MD_ESCAPE_RE = re.compile(r"\\([_*$#\[\](){}<>`|~^.!+\-])")

rows = []

def clean_chunk_text(text: str) -> str:
    text = _HTML_TAG_RE.sub(" ", str(text))
    text = _MD_ESCAPE_RE.sub(r"\1", text)
    # Fix common OCR typo in Ascend manuals where C0 (zero) was extracted as CO (letter O)
    text = re.sub(r"\bCO\b(?=\s+(?:is|needs|divides|=|closely))", "C0", text)
    return text.replace("**", "").strip()

if LOAD_PRECOMPUTED_CHUNKS and chunk_dataset_path and chunk_dataset_path.exists():
    print(f"[INFO] Fast-path: Loading pre-computed atomic chunks from {chunk_dataset_path.name}...")
    if chunk_dataset_path.suffix == ".parquet":
        raw_df = pd.read_parquet(chunk_dataset_path)
    else:
        raw_df = pd.read_json(chunk_dataset_path)

    for i, r in raw_df.iterrows():
        rows.append({
            "id": str(r["chunk_id"]),
            "text": str(r["content"]),
            "doc": str(r["doc_name"]),
            "chunk": int(i),
            "section": str(r.get("breadcrumb", "")),
            "is_table": bool(r.get("is_table", False)),
        })
    print(f"[OK] Loaded {len(rows):,} pre-computed chunks directly from disk.")
else:
    print(f"[INFO] Computing atomic table-aware chunks from {len(md_paths)} Markdown manuals...")
    for md_path in md_paths:
        doc_slug = re.sub(r"[^\w\-.]", "_", md_path.stem.strip())
        text = md_path.read_text(encoding="utf-8")
        chunks = strategy_atomic_table_aware(text, doc_name=md_path.stem)
        for i, c in enumerate(chunks):
            rows.append({
                "id": f"{doc_slug}_{i:04d}",
                "text": c["text"],
                "doc": md_path.stem,
                "chunk": i,
                "section": c["metadata"]["header_path"],
                "is_table": c["metadata"]["is_table"],
            })
    print(f"[OK] Generated {len(rows):,} atomic chunks from {len(md_paths)} documents.")

for idx in range(len(rows)):
    rows[idx]["text"] = clean_chunk_text(rows[idx]["text"])
    rows[idx]["section"] = clean_chunk_text(rows[idx]["section"])

    # Attach preceding table title/caption to atomic table chunks
    if rows[idx]["is_table"] and idx > 0 and rows[idx]["doc"] == rows[idx - 1]["doc"]:
        prev_lines = [ln.strip() for ln in rows[idx - 1]["text"].splitlines() if ln.strip() and not ln.startswith("[Doc:")]
        if prev_lines:
            caption_context = "\n".join(prev_lines[-2:])  # Grab the last 2 lines (e.g. Table title + intro)
            rows[idx]["text"] = f"{rows[idx]['section']}\n{caption_context}\n\n" + rows[idx]["text"].split("\n\n", 1)[-1]

table_chunks = sum(1 for r in rows if r["is_table"])
print(f"[SUMMARY] Total chunks: {len(rows):,} | Specification table chunks: {table_chunks:,} ({table_chunks / len(rows):.1%})")

[INFO] Fast-path: Loading pre-computed atomic chunks from huawei_ascend_rag_chunks_atomic.parquet...
[OK] Loaded 4,822 pre-computed chunks directly from disk.
[SUMMARY] Total chunks: 4,822 | Specification table chunks: 737 (15.3%)


# Indexação Vetorial com ChromaDB e nomic-embed-text

Criação da coleção vetorial persistente `"huawei_ascend_atomic_col"` usando embeddings densos do modelo `nomic-embed-text` via API REST do Ollama. A indexação e realizada em lotes de 64 chunks com barra de progresso `tqdm` e verificação de cache para evitar re-indexações desnecessárias.

In [26]:
def embed(texts, prefix="search_document: ", model=EMB_MODEL):
    prefixed = [f"{prefix}{t}" for t in texts]
    r = requests.post(f"{OLLAMA_URL}/api/embed", json={"model": model, "input": prefixed})
    r.raise_for_status()
    return r.json()["embeddings"]

def build_index(rows, name=COLLECTION_NAME, persist_dir=CHROMA_DIR, batch=64, rebuild=False):
    """Build or load a ChromaDB PersistentClient index with batching and progress tracking."""
    store = chromadb.PersistentClient(path=persist_dir)

    if rebuild:
        try:
            store.delete_collection(name)
            print(f"[INFO] Deleted existing collection: {name}")
        except Exception:
            pass

    col = store.get_or_create_collection(
        name,
        metadata={"hnsw:space": "cosine", "embedding_model": EMB_MODEL},
    )

    current_count = col.count()
    if current_count == len(rows):
        print(f"[OK] Collection '{name}' already indexed with {current_count:,} chunks. Skipping re-indexing.")
        return col

    print(f"[INFO] Indexing {len(rows):,} chunks into '{name}' (batch size = {batch})...")
    for i in tqdm(range(0, len(rows), batch), desc="Indexing Chunks", unit="batch"):
        part = rows[i : i + batch]
        col.upsert(
            ids=[r["id"] for r in part],
            documents=[r["text"] for r in part],
            embeddings=embed([r["text"] for r in part]),
            metadatas=[
                {
                    "doc": r["doc"],
                    "chunk": r["chunk"],
                    "section": r["section"],
                    "is_table": r["is_table"],
                }
                for r in part
            ],
        )

    print(f"[OK] Successfully built index! Total items in collection: {col.count():,}")
    return col

def retrieve(col, question, k=4):
    """Retrieve top-k chunks with cosine similarity."""
    assert col.metadata.get("embedding_model") == EMB_MODEL, "Embedding model mismatch"
    res = col.query(
        query_embeddings=embed([question]),
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {
            "id": i,
            "text": t,
            "doc": m["doc"],
            "section": m["section"],
            "is_table": m.get("is_table", False),
            "distance": d,
        }
        for i, t, m, d in zip(
            res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0]
        )
    ]

def sources_used(hits):
    """Format unique citations as [<doc> · <section>]."""
    seen, out = set(), []
    for h in hits:
        key = f"[{h['doc']} · {h['section']}]"
        if key not in seen:
            seen.add(key)
            out.append(key)
    return out

In [27]:
col = build_index(rows, name=COLLECTION_NAME, batch=64, rebuild=False)

[OK] Collection 'huawei_ascend_atomic_col' already indexed with 4,822 chunks. Skipping re-indexing.


# Motor de Geração RAG e Formatação de Contexto

Configuração do prompt de sistema adaptado ao ecossistema Huawei Ascend e função `ask_rag()` que orquestra a recuperação vetorial, montagem de contexto estruturado e chamada do modelo `qwen2.5:3b`.

In [28]:
SYSTEM_ANSWER = (
    "You are a technical assistant specializing in Huawei AI hardware (Ascend 310, Atlas 200 DK) "
    "and the CANN/DDK software ecosystem (ATC, OMG, AscendCL, AIPP, TE/TVM).\n"
    "Answer the user's question strictly using the context provided below. "
    "Be precise with parameter names, compilation flags, data types, and hardware specifications.\n"
    "If the information is not present in the context, state objectively that you do not have enough information."
)

def format_context(hits):
    """Assemble formatted retrieved chunks with document header metadata."""
    parts = []
    for h in hits:
        table_tag = " [Table Chunk]" if h.get("is_table") else ""
        parts.append(f"Document: {h['doc']} (Section: {h['section']}){table_tag}\nContent:\n{h['text']}")
    return "\n\n---\n\n".join(parts)

def ask_rag(question, k=4):
    """Full RAG loop: dense retrieval + context formatting + LLM generation."""
    t0 = time.time()
    hits = retrieve(col, question, k=k)
    t_retrieval = time.time() - t0

    context = format_context(hits)
    user_prompt = f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    t1 = time.time()
    response = client.chat.completions.create(
        model=GEN_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_ANSWER},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.1,
    )
    t_gen = time.time() - t1

    answer = response.choices[0].message.content
    sources = sources_used(hits)

    return {
        "answer": answer,
        "sources": sources,
        "hits": hits,
        "latency_retrieval_s": round(t_retrieval, 3),
        "latency_gen_s": round(t_gen, 3),
        "latency_total_s": round(t_retrieval + t_gen, 3),
    }

# Validação com Suíte Smoke-Test (5 Consultas Técnicas de Domínio)

Execução de 5 consultas técnicas representativas cobrindo os pilares do ecossistema Huawei Ascend para inspecionar qualidade de recuperação, presenca de tabelas e respostas do LLM:
1. **Exact Flag / Compiler:** `--input_fp16_nodes` no ATC
2. **Hardware Revisions / Specs:** Diferenças entre IT21DMDA e IT21VDMB
3. **AIPP Preprocessing:** Coeficientes de matriz CSC YUV420SP -> RGB888
4. **Troubleshooting / System Error:** Erro `CheckPartitionSpace partition space check failed`
5. **Tensor Architecture / Memory Layout:** Formato 5D NC1HWC0 e significado de C0

In [29]:
SMOKE_TEST_QUERIES = [
    {
        "id": "ST1",
        "domain": "Exact Flag / Compiler",
        "target_doc": "ATCToolInstructions.md",
        "query": "What does the --input_fp16_nodes parameter do in the ATC tool?",
    },
    {
        "id": "ST2",
        "domain": "Hardware Revisions / Specs",
        "target_doc": "Huawei_Atlas_200_DK_developer_kit_Technical_White_Paper__Model_3000__08.md",
        "query": "What are the main hardware differences between IT21DMDA and IT21VDMB board versions of the Atlas 200 DK?",
    },
    {
        "id": "ST3",
        "domain": "AIPP Preprocessing",
        "target_doc": "ATCToolInstructions.md / Atlas200AIAcceleratorModule...",
        "query": "Which CSC matrix coefficients are used in AIPP to convert YUV420SP_U8 to RGB888_U8 for JPEG images?",
    },
    {
        "id": "ST4",
        "domain": "Troubleshooting / System Error",
        "target_doc": "EnvironmentDeploymentGuide.md",
        "query": "What causes the error 'CheckPartitionSpace partition space check failed' during Atlas 200 DK driver upgrade and how to fix it?",
    },
    {
        "id": "ST5",
        "domain": "Tensor Architecture / Memory Layout",
        "target_doc": "ATCToolInstructions.md / TBECustomOperator...",
        "query": "How is the NC1HWC0 5D tensor format structured and what does C0 represent for FP16 and INT8?",
    },
]

print(f"Running smoke test on {len(SMOKE_TEST_QUERIES)} Huawei Ascend domain queries...\n")

for item in SMOKE_TEST_QUERIES:
    print("=" * 90)
    print(f"[{item['id']}] Domain: {item['domain']}")
    print(f"Target Doc: {item['target_doc']}")
    print(f"Query: {item['query']}")
    print("-" * 90)

    res = ask_rag(item["query"], k=3)

    print(f"Answer:\n{res['answer']}\n")
    print(f"Latencies: Retrieval={res['latency_retrieval_s']}s | Generation={res['latency_gen_s']}s | Total={res['latency_total_s']}s")
    print("Retrieved Sources:")
    for h in res["hits"]:
        tbl_info = " [TABLE]" if h.get("is_table") else ""
        print(f"  - doc={h['doc']} | dist={h['distance']:.4f}{tbl_info} | section={h['section']}")
    print()

Running smoke test on 5 Huawei Ascend domain queries...

[ST1] Domain: Exact Flag / Compiler
Target Doc: ATCToolInstructions.md
Query: What does the --input_fp16_nodes parameter do in the ATC tool?
------------------------------------------------------------------------------------------
Answer:
Based on the provided context, the information about the `--input_fp16_nodes` parameter in the ATC (Atlas Tool Chain) tool is not available. The context provided focuses on the Atlas 200 AI Accelerator Module Application Software Development Guide, specifically detailing class member functions, VENC function interfaces, and auxiliary APIs. There is no mention of the `--input_fp16_nodes` parameter or its functionality. Therefore, I do not have enough information to answer the question.

Latencies: Retrieval=2.294s | Generation=2.721s | Total=5.015s
Retrieved Sources:
  - doc=Atlas200AIAcceleratorModuleApplicationSoftwareDevelopmentGuide_Models3000_ | dist=0.3264 [TABLE] | section=[Doc: Atlas200A

# Tecnicas de Pre-Consulta para Otimizacao do RAG

Implementacao das 4 variantes de recuperacao pre-consulta para avaliacao comparativa:
- **Baseline:** Busca direta pelo texto original da pergunta.
- **Step-Back Prompting:** Abstracao da pergunta para recuperar conceitos fundamentais de alto nivel.
- **Multi-Query / RAG-Fusion:** Geracao de multiplas reformulacoes com fusao de rankings via Reciprocal Rank Fusion (RRF).
- **RAG com Memoria:** Reescreve perguntas dependentes de contexto historico em consultas completas e independentes.

In [30]:
# ============== PARAMETERS CONFIGURATION ==============
K            = 4      # chunks entregues ao gerador
N_RUNS       = 1      # execuções por variante
SEED_BASE    = 42     # seed para reprodutibilidade
GEN_TEMP     = 0.3    # temperatura da resposta final
REWRITE_TEMP = 0.3    # temperatura das chamadas de reescrita
N_VARIANTS   = 5      # reformulações no Multi-Query
RRF_C        = 60     # constante do Reciprocal Rank Fusion
MEMORY_DISTRACTOR = True
FORCE        = True
# =======================================================

class Trace:
    def __init__(self):
        self.llm_calls = 0
        self.embed_calls = 0
        self.queries = []

def llm(messages, tr, temperature, seed):
    tr.llm_calls += 1
    r = client.chat.completions.create(model=GEN_MODEL, messages=messages,
                                       temperature=temperature, seed=seed)
    return r.choices[0].message.content.strip()

def search(col, query, tr, k=K):
    tr.embed_calls += 1
    tr.queries.append(query)
    res = col.query(query_embeddings=embed([query]), n_results=k,
                    include=["documents", "metadatas", "distances"])
    return [
        {
            "id": i,
            "text": t,
            "doc": m["doc"],
            "section": m["section"],
            "is_table": m.get("is_table", False),
            "distance": d,
        }
        for i, t, m, d in zip(
            res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0]
        )
    ]

def rrf(rankings, c=RRF_C):
    """Reciprocal Rank Fusion"""
    fused = defaultdict(float)
    for ranking in rankings:
        for rank, cid in enumerate(ranking, 1):
            fused[cid] += 1.0 / (c + rank)
    return sorted(fused.items(), key=lambda x: -x[1])

def dedupe_topk(hits, k=K):
    best = {}
    for h in hits:
        if h["id"] not in best or h["distance"] < best[h["id"]]["distance"]:
            best[h["id"]] = h
    return sorted(best.values(), key=lambda h: h["distance"])[:k]

def clean_lines(text, original, n):
    out = []
    for line in text.splitlines():
        line = re.sub(r"^\s*(?:[-*•]|\d+[.)])\s*", "", line).strip().strip('"')
        if line and line.lower() != original.lower() and len(line) < 300 and line not in out:
            out.append(line)
    return out[:n]

def generate_answer(question, hits, tr, seed):
    messages = [
        {"role": "system", "content": SYSTEM_ANSWER},
        {"role": "user", "content": f"Context:\n{format_context(hits)}\n\nQuestion: {question}"},
    ]
    return llm(messages, tr, GEN_TEMP, seed)

def r_baseline(q, col, tr, seed):
    return q["question"], search(col, q["question"], tr)

### Step-Back Prompting (Abstracao de Conceito)

In [31]:
STEPBACK_SYSTEM = (
    "You reformulate questions for searching Huawei AI accelerator technical documentation (Ascend 310, Atlas 200 DK, CANN, ATC). "
    "Given a specific question, write ONE broader, more general (step-back) question about the underlying concept, architecture, parameter, or technical workflow. "
    "Reply only with the general question.\n\n"
    "Example 1\nQuestion: What does the --input_fp16_nodes flag do during model conversion with ATC?\n"
    "General question: What are the precision control and input data type parameters in the ATC compiler?\n\n"
    "Example 2\nQuestion: How do I configure matrix_r0c0 to convert YUV420SP to RGB888 in the AIPP file?\n"
    "General question: How does the Color Space Conversion (CSC) matrix work in AIPP preprocessing?\n\n"
    "Example 3\nQuestion: Why does the CheckPartitionSpace error occur when upgrading the Atlas 200 DK board driver?\n"
    "General question: What are the disk partitioning requirements and driver upgrade procedures on the Atlas 200 DK?"
)

def r_stepback(q, col, tr, seed):
    out = llm([{"role": "system", "content": STEPBACK_SYSTEM},
              {"role": "user", "content": f"Question: {q['question']}\nGeneral question:"}],
              tr, REWRITE_TEMP, seed)
    lines = [l.strip() for l in out.splitlines() if l.strip()]
    step_q = lines[0] if lines else q["question"]
    hits = search(col, q["question"], tr) + search(col, step_q, tr)
    return q["question"], dedupe_topk(hits, K)

### Multi-Query / RAG-Fusion (Reformulacao com Sinonimos)

In [32]:
def r_multiquery(q, col, tr, seed):
    system = (
        f"Generate {N_VARIANTS} different technical reformulations of the user's question for searching Ascend and CANN hardware manuals, "
        "using synonyms, alternative phrasing, and technical variations. "
        "Write one per line, without numbering or explanations."
    )
    out = llm([{"role": "system", "content": system}, {"role": "user", "content": q["question"]}],
              tr, REWRITE_TEMP, seed)
    queries = [q["question"]] + clean_lines(out, q["question"], N_VARIANTS)
    rankings, pool = [], {}
    for qq in queries:
        hs = search(col, qq, tr)
        rankings.append([h["id"] for h in hs])
        for h in hs:
            if h["id"] not in pool or h["distance"] < pool[h["id"]]["distance"]:
                pool[h["id"]] = h
    hits = []
    for cid, score in rrf(rankings)[:K]:
        h = dict(pool[cid])
        h["rrf"] = round(score, 5)
        hits.append(h)
    return q["question"], hits

### RAG com Memoria (Perguntas de Acompanhamento)

In [33]:
DISTRACTOR = [
    {
        "role": "user",
        "content": "What is the INT8 computing power of the Ascend 310 Processor?",
    },
    {
        "role": "assistant",
        "content": "The Ascend 310 Processor delivers up to 22 TOPS in INT8 with a typical power consumption of 8W.",
    },
]

MEMORY_SYSTEM = (
    "Given the conversation history and a follow-up question about Huawei Ascend hardware and software, "
    "rewrite the follow-up question so that it is standalone and self-contained. "
    "If it is already standalone, repeat it without changes. Reply only with the rewritten question."
)

def r_memory(q, col, tr, seed):
    history = q.get("history") or (DISTRACTOR if MEMORY_DISTRACTOR else [])
    if not history:
        return q["question"], search(col, q["question"], tr)
    hist = "\n".join(f"{'User' if t['role'] == 'user' else 'Assistant'}: {t['content']}" for t in history[-4:])
    out = llm([{"role": "system", "content": MEMORY_SYSTEM},
               {"role": "user", "content": f"History:\n{hist}\n\nFollow-up question: {q['question']}\n\nStandalone question:"}],
               tr, REWRITE_TEMP, seed)
    lines = [l.strip() for l in out.splitlines() if l.strip()]
    standalone = lines[0] if lines else q["question"]
    return standalone, search(col, standalone, tr)

VARIANTS = {"baseline": r_baseline, "stepback": r_stepback, "multiquery": r_multiquery, "memory": r_memory}

# Suíte de Avaliacao Comparativa e Exportacao de Resultados

Definicao das perguntas de teste do dominio Huawei Ascend com verificacao de palavras-chave obrigatorias no gabarito (`must_contain`) e execucao comparativa das 4 variantes.

In [34]:
os.makedirs("results", exist_ok=True)

def norm(s):
    import unicodedata
    s = unicodedata.normalize("NFKD", str(s).lower())
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s.replace("*", ""))

def Q(id, tier, question, must_contain, gold_docs, expect_refusal=False, history=None):
    return {
        "id": id,
        "tier": tier,
        "question": question,
        "must_contain": must_contain,
        "gold_docs": gold_docs,
        "expect_refusal": expect_refusal,
        "history": history or [],
    }

ATC_DOC = "ATCToolInstructions"
WP_DOC  = "Huawei_Atlas_200_DK_developer_kit_Technical_White_Paper__Model_3000__08"
ENV_DOC = "EnvironmentDeploymentGuide"
DEV_DOC = "Atlas200AIAcceleratorModuleApplicationSoftwareDevelopmentGuide_Models3000_"
TBE_DOC = "TBECustomOperatorDevelopmentGuide_CLI_"

QUESTIONS = [
    # ---------------- direct (exact parameters & specs) ----------------
    Q("q1", "direct", "What does the --input_fp16_nodes parameter do in the ATC tool?",
      ["fp16", "input_fp16_nodes"], [ATC_DOC]),
    Q("q2", "direct", "What are the main hardware differences between IT21DMDA and IT21VDMB board versions of the Atlas 200 DK?",
      ["it21dmda|it21vdmb"], [WP_DOC]),
    Q("q3", "direct", "What causes the error 'CheckPartitionSpace partition space check failed' during Atlas 200 DK driver upgrade?",
      ["checkpartitionspace|partition|space"], [ENV_DOC]),
    Q("q4", "direct", "How is the NC1HWC0 5D tensor format structured and what does C0 represent for FP16?",
      ["16", "c0|nc1hwc0"], [ATC_DOC, TBE_DOC]),

    # ---------------- multi: multiple parameters or tables ----------------
    Q("q5", "multi", "Which CSC matrix coefficients are used in AIPP to convert YUV420SP_U8 to RGB888_U8 for JPEG images?",
      ["matrix_r0c0|csc|yuv420sp"], [ATC_DOC, DEV_DOC]),
    Q("q6", "multi", "What is the typical power consumption and power input voltage range of the Atlas 200 DK?",
      ["voltage|12v|24v|power"], [WP_DOC]),

    # ---------------- paraphrase: vocabulary gap ----------------
    Q("q7", "paraphrase", "How do I normalize mean and variance before model execution in hardware?",
      ["var_reci_chn|mean|aipp"], [ATC_DOC]),
    Q("q8", "paraphrase", "What should I do if the Atlas board cannot establish connection to the Ubuntu server via USB?",
      ["usb|ip|connect"], [ENV_DOC]),

    # ---------------- followup: depends on conversation history ----------------
    Q("q9", "followup", "And what is its value for INT8 data types?",
      ["32", "c0|int8"], [ATC_DOC, TBE_DOC],
      history=[{"role": "user", "content": "What is the value of C0 for FP16 in NC1HWC0 format?"},
               {"role": "assistant", "content": "For FP16, C0 equals 16."}]),

    # ---------------- none: out of domain refusal ----------------
    Q("q10", "none", "What is the GPU clock speed of Nvidia RTX 4090?",
      [], [], expect_refusal=True),
]

QUESTIONS.sort(key=lambda q: int(q["id"][1:]))

with open("results/questions.json", "w", encoding="utf-8") as f:
    json.dump(QUESTIONS, f, ensure_ascii=False, indent=2)

print(f"[OK] Benchmark suite: {len(QUESTIONS)} questions configured for Ascend domain evaluation.")

[OK] Benchmark suite: 10 questions configured for Ascend domain evaluation.


In [35]:
REFUSAL_RE = re.compile(
    r"do not have (enough|sufficient) information|don't have (enough|sufficient) information"
    r"|insufficient information|not enough information|cannot (answer|determine|find)"
    r"|does not (contain|mention|provide|specify)|not (mentioned|provided|specified|found|available) in the (provided )?context"
    r"|no information (available|provided|found)"
    r"|nao (possui|possuo|tenho|encontr|ha informac|consta)|informacoes suficientes",
    re.I,
)

def has_kw(text_norm, kw):
    return any(norm(alt).strip() in text_norm for alt in kw.split("|"))

def grade(answer, q):
    a = norm(answer)
    refused = bool(REFUSAL_RE.search(a))
    if q["expect_refusal"]:
        return refused, refused
    return all(has_kw(a, kw) for kw in q["must_contain"]), refused

def run_variant(label, fn, questions, run, col):
    path = f"results/{label}_r{run}.csv"
    if os.path.exists(path) and not FORCE:
        return pd.read_csv(path)
    rows = []
    for qi, q in enumerate(questions):
        tr, seed = Trace(), SEED_BASE + 1000 * run + qi
        t0 = time.time()
        q_used, hits = fn(q, col, tr, seed)
        answer = generate_answer(q_used, hits, tr, seed)
        latency = time.time() - t0

        correct, refused = grade(answer, q)
        docs = {h["doc"] for h in hits}
        recall = (sum(d in docs for d in q["gold_docs"]) / len(q["gold_docs"])) if q["gold_docs"] else np.nan
        ctx = norm(format_context(hits))
        ctx_keys = all(has_kw(ctx, kw) for kw in q["must_contain"]) if q["must_contain"] else np.nan

        rows.append({
            "label": label, "run": run, "id": q["id"], "tier": q["tier"], "question": q["question"],
            "question_used": q_used, "queries": json.dumps(tr.queries, ensure_ascii=False),
            "retrieved": " | ".join(f"{h['id']}({h['distance']:.3f})" for h in hits),
            "gold_docs": ";".join(q["gold_docs"]), "gold_recall": recall, "ctx_has_keys": ctx_keys,
            "answer": answer, "correct": correct, "refused": refused,
            "llm_calls": tr.llm_calls, "embed_calls": tr.embed_calls, "latency_s": round(latency, 2),
        })
    df = pd.DataFrame(rows)
    df.to_csv(path, index=False)
    return df

In [36]:
# Evaluates variants on questions.json (optional quick run or batch evaluation)
RUN_FULL_EVALUATION = False  # Set to True when ready to execute all variants

if RUN_FULL_EVALUATION:
    with open("results/questions.json", encoding="utf-8") as f:
        questions = json.load(f)

    # Warmup call
    llm([{"role": "user", "content": "ok"}], Trace(), 0.0, 0)
    embed(["ok"])

    for label, fn in VARIANTS.items():
        for run in range(1, N_RUNS + 1):
            t0 = time.time()
            df = run_variant(label, fn, questions, run, col)
            print(f"{label:<11} r{run}  acc={df['correct'].mean():.0%}  llm_calls/pergunta={df['llm_calls'].mean():.1f}  ({time.time()-t0:.0f}s)")
        pd.concat([pd.read_csv(f"results/{label}_r{r}.csv") for r in range(1, N_RUNS + 1)]) \
          .to_csv(f"results/results_{label}.csv", index=False)

    try:
        ollama_version = requests.get(f"{OLLAMA_URL}/api/version", timeout=3).json().get("version")
    except Exception:
        ollama_version = None
    with open("results/run_config.json", "w", encoding="utf-8") as f:
        json.dump({"gen_model": GEN_MODEL, "emb_model": EMB_MODEL, "ollama_version": ollama_version,
                   "k": K, "n_runs": N_RUNS, "seed_base": SEED_BASE, "gen_temp": GEN_TEMP,
                   "rewrite_temp": REWRITE_TEMP, "n_variants": N_VARIANTS, "rrf_c": RRF_C,
                   "memory_distractor": MEMORY_DISTRACTOR, "n_chunks": len(rows), "n_questions": len(questions),
                   "date": time.strftime("%Y-%m-%d %H:%M")}, f, indent=2, ensure_ascii=False)
    print("\n[OK] Comparative results exported to results/")
else:
    print("[INFO] Set RUN_FULL_EVALUATION = True to execute the complete variant benchmark across all variants.")

[INFO] Set RUN_FULL_EVALUATION = True to execute the complete variant benchmark across all variants.
